# Week 2 — Day 4: Fusion Visualization

## Project
Contextual Predictive Maintenance (IoT Edge AI)

## Internship
Infotact DS/ML Internship

## Objective
Visualize the fused IoT and contextual dataset using correlation analysis, interaction plots and exploratory visualizations to better understand feature relationships.

## Workflow
- Load fused dataset
- Correlation analysis
- Interaction plots
- Dataset profiling
- Statistical summary

In [ ]:
"""
Week 2 - Day 4
Fusion Visualization Notebook
=============================

Deep visualization of fused IoT + Context dataset.
"""

import os
import sys
import warnings

warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")

sys.path.insert(0, src_path)

from external_data.data_fusion import (
    create_fused_dataset,
    get_fused_arrays
)

plt.style.use("seaborn-v0_8")

print("✅ Environment Ready")

In [ ]:
dataset_path = os.path.abspath(
    os.path.join(
        project_root,
        "data",
        "ai4i2020.csv"
    )
)

fused_df = create_fused_dataset(dataset_path)

X, y, feature_names = get_fused_arrays(fused_df)

print(f"Dataset Shape: {fused_df.shape}")
print(f"Features: {len(feature_names)}")
print(f"Failures: {y.sum()}")

In [ ]:
feature_groups = {
    "Weather": [
        "ambient_temp_c",
        "humidity_pct",
        "wind_speed_kmh",
        "air_pressure_hpa",
        "weather_encoded"
    ],
    "Factory Load": [
        "factory_load_pct",
        "machine_util_pct",
        "shift_encoded",
        "is_weekend",
        "maintenance_flag",
        "production_rate"
    ]
}

print("=" * 50)
print("FEATURE GROUP SUMMARY")
print("=" * 50)

for group, cols in feature_groups.items():

    available = [
        c for c in cols
        if c in fused_df.columns
    ]

    print(f"\n{group}")

    for col in available:
        print(f"   • {col}")

In [ ]:
numeric_df = fused_df.select_dtypes(
    include=np.number
)

corr_matrix = numeric_df.corr()

plt.figure(figsize=(14, 10))

sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0
)

plt.title(
    "Fusion Dataset Correlation Matrix",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()

plt.show()

In [ ]:
candidate_features = [
    "Torque [Nm]",
    "Tool wear [min]",
    "ambient_temp_c",
    "factory_load_pct",
    "Machine failure"
]

available_features = [
    c for c in candidate_features
    if c in fused_df.columns
]

pair_df = fused_df[
    available_features
].copy()

if "Machine failure" in pair_df.columns:

    pair_df["Machine failure"] = pair_df[
        "Machine failure"
    ].map({
        0: "No Failure",
        1: "Failure"
    })

sample_size = min(1500, len(pair_df))

sample_df = pair_df.sample(
    sample_size,
    random_state=42
)

sns.pairplot(
    sample_df,
    hue="Machine failure"
)

plt.show()

In [ ]:
rows_to_plot = 500

available_plots = []

if "Torque [Nm]" in fused_df.columns:
    available_plots.append(
        ("Torque Signal", "Torque [Nm]")
    )

if "ambient_temp_c" in fused_df.columns:
    available_plots.append(
        ("Ambient Temperature", "ambient_temp_c")
    )

if "factory_load_pct" in fused_df.columns:
    available_plots.append(
        ("Factory Load", "factory_load_pct")
    )

n_plots = len(available_plots)

fig, axes = plt.subplots(
    n_plots,
    1,
    figsize=(14, 4 * n_plots)
)

if n_plots == 1:
    axes = [axes]

for ax, (title, col) in zip(
    axes,
    available_plots
):
    ax.plot(
        fused_df[col].iloc[:rows_to_plot]
    )
    ax.set_title(title)

plt.tight_layout()
plt.show()

In [ ]:
summary_cols = [
    "Torque [Nm]",
    "Tool wear [min]",
    "ambient_temp_c",
    "factory_load_pct"
]

available_cols = [
    c for c in summary_cols
    if c in fused_df.columns
]

summary = fused_df.groupby(
    "Machine failure"
)[available_cols].mean()

print(summary)

In [ ]:
print("=" * 60)
print("WEEK 2 DAY 4 - COMMIT 1 COMPLETE")
print("Fusion Visualization Notebook Ready")
print("=" * 60)